# 03 - Model Training and Performance Comparison

This notebook documents the machine learning stage of the **Viral Outbreak Early Warning System** project.

The objective is to train and compare supervised classification models capable of predicting whether a simulated region will experience outbreak risk within the next 7 days.

The target variable is:

`outbreak_7d`

## 1. Modeling Objective

The goal of this stage is to evaluate whether epidemiological, environmental, mobility and healthcare-related variables can be used to predict future outbreak risk.

This is treated as a binary classification problem:

- `0`: no outbreak expected within the next 7 days
- `1`: outbreak risk expected within the next 7 days

## 2. Why Use a Temporal Train/Test Split?

In epidemiological surveillance, using a random train/test split can create unrealistic results because future observations may leak into the training set.

To make the evaluation more realistic, the project uses a **temporal holdout split**:

- earlier dates are used for training
- later dates are used for testing

This better approximates a real-world early warning system, where the model is trained on historical data and evaluated on future observations.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Check project structure
!pwd
!ls

## 3. Load the Generated Dataset

In [ ]:
DATA_PATH = Path("data/raw/epiwatch_epidemiological_surveillance_dataset.csv")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

df.head()

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

## 4. Target Distribution

Before training, it is important to verify that the target contains both classes.

In [ ]:
df["outbreak_7d"].value_counts(normalize=True)

## 5. Run the Training Script

The training pipeline is implemented in:

`src/train_model.py`

This script:

1. Loads the generated dataset
2. Applies a temporal train/test split
3. Preprocesses numeric and categorical features
4. Trains multiple machine learning models
5. Evaluates model performance
6. Saves the best model and performance tables

In [ ]:
!python src/train_model.py

## 6. Load Model Performance Results

The training script saves the model comparison table in:

`reports/tables/model_performance_metrics.csv`

In [ ]:
METRICS_PATH = Path("reports/tables/model_performance_metrics.csv")

metrics = pd.read_csv(METRICS_PATH)
metrics

## 7. Model Comparison

The models are compared using several metrics:

- **Accuracy**: overall proportion of correct predictions
- **Precision**: proportion of predicted outbreaks that were true outbreaks
- **Recall**: proportion of actual outbreaks detected by the model
- **F1-score**: harmonic mean of precision and recall
- **ROC AUC**: discrimination ability across thresholds
- **PR AUC**: precision-recall performance, useful when outbreak risk is imbalanced

For early warning systems, **recall** and **PR AUC** are especially important because missing an outbreak can be more costly than generating a false alarm.

In [ ]:
display_columns = [
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "pr_auc"
]

metrics[display_columns]

## 8. Visual Comparison of Model Performance

In [ ]:
performance_metrics = ["accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]

metrics_plot = metrics.set_index("model")[performance_metrics]

plt.figure(figsize=(11, 6))
metrics_plot.plot(kind="bar")

plt.title("Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

## 9. Ranking by Precision-Recall AUC

Precision-Recall AUC is particularly relevant in outbreak prediction because it focuses on performance for the positive class: future outbreak risk.

In [ ]:
ranked_by_pr_auc = metrics.sort_values("pr_auc", ascending=False)

ranked_by_pr_auc[["model", "pr_auc", "recall", "f1_score", "roc_auc"]]

In [ ]:
plt.figure(figsize=(8, 5))

ranked = metrics.sort_values("pr_auc", ascending=True)
plt.barh(ranked["model"], ranked["pr_auc"])

plt.title("Model Ranking by Precision-Recall AUC")
plt.xlabel("PR AUC")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.tight_layout()
plt.show()

## 10. Confusion Matrix Values

The training pipeline also stores true positives, false positives, true negatives and false negatives for each model.

In [ ]:
confusion_columns = [
    "model",
    "true_negatives",
    "false_positives",
    "false_negatives",
    "true_positives"
]

metrics[confusion_columns]

## 11. Best Model Selection

The best model is selected using **PR AUC**.

This is a suitable choice for outbreak early warning because the model should perform well at identifying future outbreak events rather than only maximizing overall accuracy.

In [ ]:
best_model = metrics.sort_values("pr_auc", ascending=False).iloc[0]

best_model

## 12. Interpretation of the Results

The model comparison provides a realistic evaluation of outbreak prediction performance.

A perfect score would be suspicious in this type of project because epidemiological data are noisy and outbreak dynamics are stochastic.

Moderate ROC AUC and PR AUC values are more realistic and suggest that the synthetic dataset contains meaningful but imperfect predictive signals.

## 13. Outputs Generated by This Stage

This notebook relies on `src/train_model.py`, which generates:

```text
reports/tables/model_performance_metrics.csv
reports/tables/classification_reports.json
models/best_outbreak_prediction_model.pkl
```

These outputs are used in later stages for final evaluation, visualization and explainability.

## 14. Conclusion

This notebook trained and compared several machine learning models for 7-day outbreak prediction.

The next step is final model evaluation, including ROC curves, Precision-Recall curves, confusion matrices and threshold sensitivity analysis.